# Week 4: Machine Learning for Forecasting
## In-Class Exercises

**Objective.** Turn a time series into a supervised learning problem without fooling yourself, fit gradient boosting to it, and find out where trees beat ETS and where they fall over.

### How this notebook works

Four parts and an appendix, each building on the one before it.

| Part | Format | Content |
| --- | --- | --- |
| 1 | Walkthrough | `mlforecast` end to end, plus the failure that motivates everything after it. |
| 2 | Blanks we fill in together | `mlforecast` vs. a hand-rolled scikit-learn pipeline on real bike-share data; dummies vs. Fourier terms. |
| 3 | On your own, ~15 min | Build the lag matrix by hand, then recursive vs. direct forecasting. |
| 4 | Walkthrough | Exogenous features: calendar facts vs. real weather, and what happens as a weather forecast degrades. |
| Appendix | Read on your own | A warning: how much evaluation methodology alone can distort apparent accuracy. |


In [ ]:
!pip install -q mlforecast lightgbm scikit-learn pandas plotly


In [ ]:
import time
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
from sklearn.base import clone
from lightgbm import LGBMRegressor
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from mlforecast.target_transforms import Differences

pio.templates.default = "plotly_white"   # try "plotly_dark", "ggplot2", "simple_white"

URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
air = (pd.read_csv(URL, parse_dates=["Month"])
         .rename(columns={"Month": "ds", "Passengers": "y"})
         .assign(unique_id="airline"))

h, m = 24, 12
train = air.iloc[:-h].copy()
test = air.iloc[-h:].copy()


def mae(y, yhat):
    return np.mean(np.abs(np.asarray(y, float) - np.asarray(yhat, float)))

def mase(y, yhat, y_train, m=12):
    y_train = np.asarray(y_train, float)
    return mae(y, yhat) / np.mean(np.abs(y_train[m:] - y_train[:-m]))


def forecast_plot(preds, title, history=train, tail=None):
    """History plus test actuals plus one line per model column in `preds`."""
    hist = history.tail(tail) if tail else history
    tidy = pd.concat([
        hist.assign(series="train").rename(columns={"y": "value"})[["ds", "series", "value"]],
        test.assign(series="test").rename(columns={"y": "value"})[["ds", "series", "value"]],
        preds.melt(id_vars="ds", value_vars=[c for c in preds if c not in ("unique_id", "ds")],
                   var_name="series", value_name="value"),
    ])
    return px.line(tidy, x="ds", y="value", color="series", title=title)


train.tail(3)


---
## Part 1. The ML framing, and the trap inside it

A forecasting model becomes a regression model the moment you write

$$y_t = f(y_{t-1}, y_{t-2}, \ldots, y_{t-m}, \text{calendar features}) + \varepsilon_t$$

`mlforecast` builds that design matrix, fits any scikit-learn-style model to it, and rolls the predictions forward recursively.

Watch the first fit. It fails in a specific, instructive way.


In [ ]:
fcst = MLForecast(
    models={"lr": LinearRegression(), "gbm": LGBMRegressor(n_estimators=200, verbosity=-1)},
    freq="MS",
    lags=[1, 12],
    lag_transforms={1: [RollingMean(window_size=12)]},
    date_features=["month"],
)
fcst.fit(train)
p_raw = fcst.predict(h)

forecast_plot(p_raw, "First attempt: no target transform").show()

for mdl in ["lr", "gbm"]:
    print(mdl, "MASE:", round(mase(test["y"], p_raw[mdl], train["y"], m), 3))


**Notice:**

1. The gradient boosting forecast flattens and never exceeds the highest value seen in training. Trees predict an average of training targets, so they cannot produce a number outside the training range.
2. Linear regression extrapolates, so `lr` often beats `gbm` here.
3. `Differences([1])` removes the trend before the model sees the data and adds it back after, which fixes this.


In [ ]:
fcst2 = MLForecast(
    models={"lr": LinearRegression(), "gbm": LGBMRegressor(n_estimators=200, verbosity=-1)},
    freq="MS",
    lags=[1, 12],
    lag_transforms={1: [RollingMean(window_size=12)]},
    date_features=["month"],
    target_transforms=[Differences([1])],     # <- the whole difference
)
fcst2.fit(train)
p = fcst2.predict(h)

forecast_plot(p, "With Differences([1])", tail=60).show()

snaive = np.tile(train["y"].to_numpy()[-m:], int(np.ceil(h / m)))[:h]
print("seasonal naive MASE:", round(mase(test["y"], snaive, train["y"], m), 3))
for mdl in ["lr", "gbm"]:
    print(f"{mdl} MASE:", round(mase(test["y"], p[mdl], train["y"], m), 3))


**Notice:** with 120 observations and one series, seasonal naive is a serious competitor. ML methods earn their keep on panels of hundreds or thousands of related series pooled into one global model - Week 5. Today is mechanics and traps, on one series at a time.

---
## Part 2. Two practical choices: implementation, and how you encode season

Daily bike-share rentals, Washington D.C., 2011-2012 (Fanaee-T & Gama, UCI). `cnt` is total rentals; `casual` and `registered` are the two rider types that sum to it. We hold out the last 28 days and never let the model see them.


In [ ]:
BIKE_URL = "https://raw.githubusercontent.com/christophM/interpretable-ml-book/master/data/bike.csv"
bike_raw = pd.read_csv(BIKE_URL, parse_dates=["dteday"])   # also used for Part 4's weather columns
bike = (bike_raw[["dteday", "cnt", "casual", "registered"]]
        .set_index("dteday").asfreq("D").interpolate().reset_index()
        .rename(columns={"dteday": "ds"}))

H, m2 = 28, 7
bk = bike.assign(unique_id="bike", y=bike["cnt"])[["unique_id", "ds", "y"]]
train_b, test_b = bk.iloc[:-H].copy(), bk.iloc[-H:].copy()

px.line(bike, x="ds", y="cnt", title="Daily bike rentals, 2011-2012").show()
bike.groupby(bike["ds"].dt.dayofweek)[["casual", "registered"]].mean().round(0)


`casual` peaks on weekends, `registered` peaks on weekdays. The two rider types pull the weekly pattern in opposite directions, and `cnt` is their sum.


### Beat 1. `mlforecast` vs. a hand-rolled scikit-learn pipeline

Same lags, same model, same 28-day forecast, two implementations.


In [ ]:
def sklearn_recursive_forecast(train_df, h, lags=(1, 7, 14)):
    s = train_df.set_index("ds")["y"]
    d = pd.DataFrame({"y": s})
    for L in lags:
        d[f"lag{L}"] = d["y"].shift(L)
    d["dow"] = d.index.dayofweek
    d = d.dropna()
    Xd, yd = d.drop(columns="y"), d["y"]
    model = LGBMRegressor(n_estimators=300, verbosity=-1).fit(Xd, yd)

    hist = list(s.to_numpy())
    future_idx = pd.date_range(s.index[-1] + pd.Timedelta(days=1), periods=h, freq="D")
    out = []
    for dt in future_idx:
        row = pd.DataFrame([{f"lag{L}": hist[-L] for L in lags} | {"dow": dt.dayofweek}])[Xd.columns]
        pr = float(model.predict(row)[0])
        out.append(pr)
        hist.append(pr)
    return pd.Series(out, index=future_idx)


t0 = time.perf_counter()
sk_fc = sklearn_recursive_forecast(train_b, H)
sk_time = time.perf_counter() - t0
print(f"sklearn hand-rolled: {sk_time:.2f}s  MAE {mae(test_b['y'], sk_fc.to_numpy()):.1f}")


In [ ]:
t0 = time.perf_counter()
fcst_bike = MLForecast(models={"gbm": LGBMRegressor(n_estimators=300, verbosity=-1)},
                       freq="D", lags=[1, 7, 14], date_features=["dayofweek"])
fcst_bike.fit(train_b)
ml_fc = fcst_bike.predict(H)
ml_time = time.perf_counter() - t0
print(f"mlforecast:          {ml_time:.2f}s  MAE {mae(test_b['y'], ml_fc['gbm'].to_numpy()):.1f}")

# rolling-origin CV, one call, no hand-written loop
cv = fcst_bike.cross_validation(train_b, n_windows=3, h=H)
print(f"mlforecast.cross_validation, 3 windows: MAE {mae(cv['y'], cv['gbm']):.1f}")


**Notice:**

1. Both produce the same forecast - `mlforecast`'s lag alignment matches what the hand-rolled loop does by construction.
2. The hand-rolled version is where the kind of leakage the appendix warns about lives: `KFold(shuffle=True)` is one keyword away and wrong, and nothing stops you from writing it. `mlforecast.cross_validation` only produces rolling-origin splits.
3. `mlforecast` costs control: a custom loss, a non-tabular feature, or a model with a different `.fit` signature is easier in the hand-rolled loop, and scikit-learn's ecosystem (`Pipeline`, `GridSearchCV`, calibration) is built for exactly that.
4. `mlforecast` costs nothing on a panel: the same `.fit(train_df)` call works whether `train_df` has one `unique_id` or ten thousand. The hand-rolled loop above only handles one series.


### Beat 2. Dummies vs. Fourier terms

Two ways to encode a seasonal period of length $p$:

- **Dummies**: $p - 1$ binary columns, one parameter per point in the cycle.
- **Fourier terms**: $\sin$/$\cos$ pairs at that period, $2K$ parameters for $K$ pairs.

At $p = 7$ the two are close enough not to matter much. Bike rentals also carry a real annual cycle, and at $p \approx 365$ the cost difference stops being close.


In [ ]:
def fourier_terms(index, period, K, prefix):
    """K sin/cos pairs at the given period, as a DataFrame aligned to index."""
    t = np.arange(len(index))
    cols = {}
    for k in range(1, K + 1):
        cols[f"{prefix}_sin{k}"] = np.sin(2 * np.pi * k * t / period)
        cols[f"{prefix}_cos{k}"] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(cols, index=index)


bike_idx = bike.set_index("ds")
tr_idx, te_idx = bike_idx.index[:-H], bike_idx.index[-H:]
n_days = len(bike_idx)

dow = pd.get_dummies(bike_idx.index.dayofweek, prefix="dow", drop_first=True).set_index(bike_idx.index).astype(float)
trend = pd.Series(np.arange(n_days), index=bike_idx.index, name="trend")

# Design A: day-of-week dummies + week-of-year dummies
woy = pd.get_dummies(bike_idx.index.isocalendar().week.astype(int), prefix="woy",
                     drop_first=True).set_index(bike_idx.index).astype(float)
X_dummy = pd.concat([dow, woy, trend], axis=1)

# Design B: day-of-week dummies + yearly Fourier
# TODO - build X_fourier the same way, but replace `woy` with 3 Fourier pairs at period 365.25.
#   Hint: fourier_terms(bike_idx.index, 365.25, K=3, prefix="yr")
X_fourier = ...


<details>
<summary><b>Show the line</b></summary>

```python
X_fourier = pd.concat([dow, fourier_terms(bike_idx.index, 365.25, K=3, prefix="yr"), trend], axis=1)
```
</details>


In [ ]:
def fit_score(X, label):
    mdl = LinearRegression().fit(X.loc[tr_idx], bike_idx["cnt"].loc[tr_idx])
    pred = mdl.predict(X.loc[te_idx])
    resid = bike_idx["cnt"].loc[tr_idx] - mdl.predict(X.loc[tr_idx])
    n_, k = len(tr_idx), X.shape[1] + 1
    aic = n_ * np.log(np.sum(resid ** 2) / n_) + 2 * k
    aicc = aic + 2 * k * (k + 1) / (n_ - k - 1)
    return {"model": label, "n_features": X.shape[1], "AICc": round(aicc, 1),
            "test MAE": round(mae(bike_idx["cnt"].loc[te_idx], pred), 1)}


pd.DataFrame([fit_score(X_dummy, "dow dummies + week-of-year dummies"),
              fit_score(X_fourier, "dow dummies + yearly Fourier (K=3)")])


**Notice:** the two designs land close together on AICc and test error, using far fewer columns for Fourier. Fourier is not a better model here, it is a cheaper one for a similar fit - and at annual periods on daily data, cheap is what keeps the design matrix estimable at all.


---
## Part 3. Building the lag matrix yourself

About 15 minutes, on the bike data from Part 2. `mlforecast` hid the machinery there; now you write it, because the bugs live in there.

**Tasks.**

1. Finish `make_supervised(y, lags)` so it returns `X` (one column per lag) and `y`, aligned, with no NaN rows.
2. Fit `Ridge` and `LGBMRegressor` on `train_b["y"]` with lags 1, 2, 7, 14.
3. Forecast 28 steps **recursively**: predict one step, append it to the history, recompute lags, repeat.
4. Forecast 28 steps **directly**: one model per horizon $k$, target shifted $k$ steps ahead, using only lags available at forecast time.
5. Compare both to seasonal naive (period 7) with MASE.

Watch the alignment in step 1. An off-by-one there shows up as suspiciously excellent accuracy - the same failure the appendix's leakage ladder demonstrates, one step removed.


In [ ]:
def make_supervised(y, lags):
    """y: 1-d array. Returns (X, target) with rows containing no NaNs."""
    s = pd.Series(np.asarray(y, float))
    X = pd.DataFrame({f"lag{L}": s.shift(L) for L in lags})
    # TODO - align the target with X, then drop rows with any NaN
    target = ...
    keep = ...
    return X[keep], target[keep]


# Quick self-check once you have it: with lags=[1], X["lag1"] should equal target shifted by one.
# make_supervised(np.arange(10), [1])


In [ ]:
# YOUR CODE HERE - tasks 2 through 5
# LAGS = [1, 2, 7, 14]; H and m2 are already defined from Part 2.


<details>
<summary><b>Solution</b></summary>

```python
def make_supervised(y, lags):
    s = pd.Series(np.asarray(y, float))
    X = pd.DataFrame({f"lag{L}": s.shift(L) for L in lags})
    target = s
    keep = X.notna().all(axis=1)
    return X[keep], target[keep]


LAGS = [1, 2, 7, 14]
ytr = train_b["y"].to_numpy()
X, tgt = make_supervised(ytr, LAGS)

models = {"ridge": Ridge().fit(X, tgt),
          "gbm": LGBMRegressor(n_estimators=300, verbosity=-1).fit(X, tgt)}

#  recursive
rec = {}
for name, mdl in models.items():
    hist = list(ytr)
    preds = []
    for _ in range(H):
        row = pd.DataFrame([[hist[-L] for L in LAGS]], columns=X.columns)
        nxt = float(mdl.predict(row)[0])
        preds.append(nxt)
        hist.append(nxt)
    rec[name] = np.array(preds)

#  direct: one model per horizon k
# Row i of Xk holds lags measured k-1 steps before the target at row i,
# so at the forecast origin the feature row is simply the last observed lags.
dir_preds = {name: np.empty(H) for name in models}
last_row = pd.DataFrame([[ytr[-L] for L in LAGS]], columns=[f"lag{L}" for L in LAGS])
for k in range(1, H + 1):
    s = pd.Series(ytr)
    Xk = pd.DataFrame({f"lag{L}": s.shift(L + k - 1) for L in LAGS})
    keep = Xk.notna().all(axis=1)
    Xk, tk = Xk[keep], s[keep]
    for name, base in models.items():
        dir_preds[name][k - 1] = clone(base).fit(Xk, tk).predict(last_row)[0]

snaive = np.tile(ytr[-m2:], int(np.ceil(H / m2)))[:H]
print("seasonal naive:", round(mase(test_b["y"], snaive, ytr, m2), 3))
for name in models:
    print(f"{name} recursive:", round(mase(test_b["y"], rec[name], ytr, m2), 3),
          " direct:", round(mase(test_b["y"], dir_preds[name], ytr, m2), 3))
```

**Notice:**

1. Recursive trains one model and feeds its own predictions back in, so errors compound with horizon.
2. Direct trains $h$ models on fewer rows each, so nothing compounds, but the $h$ models are not constrained to agree with each other.
3. Ridge beats `gbm` and seasonal naive here; `gbm` loses to seasonal naive under both framings - the same extrapolation failure as Part 1, on a series with a real trend.
</details>


---
## Part 4. Exogenous features

Two kinds of predictor beyond the target's own history:

- **Known in advance**: calendar facts about the forecast date itself - day of week, holiday, whether it is a working day. Free; no forecast required.
- **Unknown in advance**: conditions on that date, like weather. Usable only if you have a forecast for them, and only as good as that forecast is.

`mlforecast` makes this distinction concrete: every non-lag feature beyond the training window has to be handed to `.predict()` explicitly, through `X_df`. There is no default that fills it in from the true future.


In [ ]:
exog_raw = (bike_raw[["dteday", "holiday", "workday", "weather", "temp", "atemp", "hum", "windspeed"]]
            .rename(columns={"dteday": "ds"})
            .set_index("ds").asfreq("D").ffill().reset_index())
exog_raw["holiday"] = (exog_raw["holiday"] == "Y").astype(int)
exog_raw["workday"] = (exog_raw["workday"] == "Y").astype(int)
exog_raw["weather"] = exog_raw["weather"].astype("category").cat.codes

full = bk.merge(exog_raw, on="ds")
train_full, test_full = full.iloc[:-H].copy(), full.iloc[-H:].copy()

LAGS_EXOG = [1, 7, 14]
CAL_COLS = ["holiday", "workday"]
WEATHER_COLS = ["temp", "atemp", "hum", "windspeed", "weather"]


def fit_predict(feature_cols):
    mdl = MLForecast(models={"gbm": LGBMRegressor(n_estimators=300, verbosity=-1)},
                     freq="D", lags=LAGS_EXOG, date_features=["dayofweek"])
    cols = ["unique_id", "ds", "y"] + feature_cols
    kw = {"static_features": []} if feature_cols else {}
    mdl.fit(train_full[cols], **kw)
    X_df = test_full[["unique_id", "ds"] + feature_cols] if feature_cols else None
    pred = mdl.predict(H, X_df=X_df) if feature_cols else mdl.predict(H)
    return mase(test_full["y"], pred["gbm"], train_full["y"], m2)


results = {
    "lags only":                          fit_predict([]),
    "+ calendar (holiday, workday)":      fit_predict(CAL_COLS),
    "+ real future weather (oracle)":     fit_predict(CAL_COLS + WEATHER_COLS),
}
pd.Series(results, name="MASE").round(3).to_frame()


**Notice:**

1. Lags alone lose to seasonal naive. Same finding as Part 1 and Part 3.
2. Calendar features help only a little: they say which day it is, not what the weather will do to demand.
3. Real future weather brings the error well below seasonal naive. Temperature is doing more work than any lag here.
4. That result needs a caveat: it uses the *true* future weather, which nobody has at the forecast origin. It is a ceiling, not an achievable forecast.


In [ ]:
rng = np.random.default_rng(0)
noise_levels = [0.0, 0.15, 0.3, 0.6, 1.0]

m_oracle = MLForecast(models={"gbm": LGBMRegressor(n_estimators=300, verbosity=-1)},
                      freq="D", lags=LAGS_EXOG, date_features=["dayofweek"])
m_oracle.fit(train_full[["unique_id", "ds", "y"] + CAL_COLS + WEATHER_COLS], static_features=[])

rows = []
for frac in noise_levels:
    fut = test_full[["unique_id", "ds"] + CAL_COLS + WEATHER_COLS].copy()
    for c in ["temp", "atemp", "hum", "windspeed"]:
        fut[c] = fut[c] + rng.normal(scale=frac * train_full[c].std(), size=len(fut))
    pred = m_oracle.predict(H, X_df=fut)
    rows.append({"weather forecast noise (x train std)": frac,
                 "MASE": mase(test_full["y"], pred["gbm"], train_full["y"], m2)})

noise_df = pd.DataFrame(rows)
px.line(noise_df, x="weather forecast noise (x train std)", y="MASE", markers=True,
        title="Forecast value decays with weather-forecast quality").show()
noise_df.round(3)


**Notice:** MASE rises as the weather readings get noisier and settles well short of the calendar-only baseline, because the categorical `weather` condition was left at its true future value in every run here - only the continuous readings (temperature, humidity, wind) were degraded. A feature is worth including only as far as your forecast of it is worth trusting.


---
## Wrap-up

1. **Trees cannot extrapolate.** Always difference or detrend a trending target.
2. **A feature has to exist at forecast time.** `lag1` is `y[t-1]`; if that will not be known until forecast time, or is defined from the target, it cannot go in the matrix.
3. **`mlforecast` buys correctness and scale; a hand-rolled pipeline buys control.** Both have a place - the panel setting next week is where `mlforecast` earns its keep.
4. **Fourier terms are a cost saving, not an accuracy trick**, and the saving grows with the period.
5. **Exogenous features can matter more than lags** - real weather mattered far more than any lag here - but only as far as you can forecast them. A feature unknown at the forecast origin needs its own forecast, with its own error, or it does not belong in `X_df`.
6. **See the appendix.** Evaluation methodology alone can make the same model, on the same task, look far better than it will actually perform.


---
## Appendix: a warning about evaluation

Four evaluations of the same forecasting task, same features, same model. Only the evaluation changes.


In [ ]:
snaive_b = np.tile(train_b["y"].to_numpy()[-m2:], int(np.ceil(H / m2)))[:H]

LAGS = [1, 7, 14]
lag_df = pd.DataFrame({"y": bk.set_index("ds")["y"]})
for L in LAGS:
    lag_df[f"lag{L}"] = lag_df["y"].shift(L)
lag_df["dow"] = lag_df.index.dayofweek
lag_df = lag_df.dropna()
X, y = lag_df.drop(columns="y"), lag_df["y"]
Xtr, ytr, Xte, yte = X.iloc[:-H], y.iloc[:-H], X.iloc[-H:], y.iloc[-H:]

gbm = LGBMRegressor(n_estimators=300, verbosity=-1)

# 1. shuffled k-fold on the lagged rows
shuffled_mae = -cross_val_score(gbm, X, y, cv=KFold(5, shuffle=True, random_state=0),
                                scoring="neg_mean_absolute_error").mean()

# 2. time-ordered folds
ts_mae = -cross_val_score(gbm, X, y, cv=TimeSeriesSplit(5),
                          scoring="neg_mean_absolute_error").mean()

# 3. fit once, then score by feeding the true lag-1/7/14 at every test date
gbm.fit(Xtr, ytr)
leaky_test_mae = mae(yte, gbm.predict(Xte))

# 4. the actual task: forecast 28 days ahead using only what a model at the
#    origin date could know - its own earlier predictions, not the true future
hist = list(ytr.to_numpy())
preds = []
for dt in yte.index:
    row = pd.DataFrame([{"lag1": hist[-1], "lag7": hist[-7], "lag14": hist[-14],
                         "dow": dt.dayofweek}])[X.columns]
    pr = float(gbm.predict(row)[0])
    preds.append(pr)
    hist.append(pr)
honest_mae = mae(yte, preds)

pd.DataFrame([
    {"evaluation": "shuffled K-fold on lagged rows", "MAE": shuffled_mae},
    {"evaluation": "TimeSeriesSplit", "MAE": ts_mae},
    {"evaluation": "true lags fed at every test date", "MAE": leaky_test_mae},
    {"evaluation": "honest 28-day recursive forecast", "MAE": honest_mae},
    {"evaluation": "seasonal naive (7-day)", "MAE": mae(test_b["y"], snaive_b)},
]).round(1)


**Notice:**

1. Shuffled K-fold scores best. A random split puts `lag1` for a test row right next to the row that produced it, in the training fold.
2. `TimeSeriesSplit` is closer, but still evaluates the model as a one-step-ahead predictor with the true lag-1 in hand - not the 28-day forecast the task actually needs.
3. Feeding the model its own true lags at every test date, instead of its own predictions, still understates the error the real forecast will have.
4. The honest recursive forecast is worse than seasonal naive. Every row above it was optimistic for a different reason.
5. `lag1` at $t$ is `y[t-1]`. If that will not be known until forecast time - or is defined from the target, like `casual + registered = cnt` - it cannot be a feature. Same rule Part 4 applies to weather.
